In [1]:
import numpy as np 
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

## Leser inn data

In [2]:
raw_demographic = pd.read_csv("raw_data/demographic.csv")
raw_hospital = pd.read_csv("raw_data/hospital.csv")
raw_physiological = pd.read_csv("raw_data/physiological.txt", sep="\t", header=0)
raw_severity = pd.read_json("raw_data/severity.json")

## Omgjøring av severity.json til dataframe

In [3]:
severity_df = raw_severity.explode(list(raw_severity.columns[2:]))
severity_df.reset_index(drop=True, inplace=True)
severity_df = severity_df.sort_values(by=['pasient_id'], ignore_index=True)

## Kombinerer til en dataframe 

In [4]:
demographic_hospital_df = raw_demographic.merge(raw_hospital, on='pasient_id', how='left')
demographic_hospital_phisiological_df = demographic_hospital_df.merge(raw_physiological, on='pasient_id')
df = demographic_hospital_phisiological_df.merge(severity_df, on='pasient_id')

Fjerner dublikater i dataframen skulle det eksistere noen

In [5]:
print(df.shape)
df.drop_duplicates(inplace=True)
print(df.shape)

(7742, 41)
(7740, 41)


## Splitting av data

In [6]:
df_train, df_val_test = train_test_split(df, test_size=0.3, shuffle=True, random_state=42)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=42)

## Numerisk data

In [7]:
df_train.dtypes

pasient_id                        object
alder                            float64
kjønn                             object
utdanning                        float64
inntekt                           object
etnisitet                         object
sykehusdød                         int64
oppholdslengde                     int64
blodtrykk                        float64
hvite_blodlegemer                float64
hjertefrekvens                   float64
respirasjonsfrekvens             float64
kroppstemperatur                 float64
lungefunksjon                    float64
serumalbumin                     float64
bilirubin                        float64
kreatinin                        float64
natrium                          float64
blod_ph                          float64
glukose                          float64
blodurea_nitrogen                float64
urinmengde                       float64
sykdomskategori_id                object
sykdomskategori                   object
dødsfall        

In [8]:
object_columns= df_train.select_dtypes(include=['object']).columns.tolist()
object_columns

['pasient_id',
 'kjønn',
 'inntekt',
 'etnisitet',
 'sykdomskategori_id',
 'sykdomskategori',
 'dødsfall',
 'sykdom_underkategori',
 'antall_komorbiditeter',
 'koma_score',
 'adl_pasient',
 'adl_stedfortreder',
 'fysiologisk_score',
 'apache_fysiologisk_score',
 'overlevelsesestimat_2mnd',
 'overlevelsesestimat_6mnd',
 'diabetes',
 'demens',
 'kreft',
 'lege_overlevelsesestimat_2mnd',
 'lege_overlevelsesestimat_6mnd',
 'dnr_status',
 'dnr_dag']

Alle "object" typer indikerer til enten string objekter, manglende data eller en blanding av datatyper. Noen av disse kolonnene vet vi skal være int. 

Setter inn dummies i disse kolonnene; 
kjønn, inntekt, etnisitet, sykdomskategori_id, sykdomskategori, sykdom_underkategori, kreft, dnr_status

In [9]:
int_cols = ['pasient_id', 'dødsfall', 'antall_komorbiditeter', 'diabetes', 'demens']
float_cols = ['koma_score', 'adl_pasient', 'adl_stedfortreder', 'fysiologisk_score', 'apache_fysiologisk_score',
              'overlevelsesestimat_2mnd', 'overlevelsesestimat_6mnd', 'lege_overlevelsesestimat_2mnd',
              'lege_overlevelsesestimat_6mnd', 'dnr_dag']

In [10]:
df_train[int_cols] = df_train[int_cols].astype(dtype=int)
df_train[float_cols] = df_train[float_cols].astype(dtype=float)

In [11]:
object_columns = df_train.select_dtypes(include='object').columns
print(list(object_columns))

['kjønn', 'inntekt', 'etnisitet', 'sykdomskategori_id', 'sykdomskategori', 'sykdom_underkategori', 'kreft', 'dnr_status']


In [12]:
df_train.dtypes

pasient_id                         int64
alder                            float64
kjønn                             object
utdanning                        float64
inntekt                           object
etnisitet                         object
sykehusdød                         int64
oppholdslengde                     int64
blodtrykk                        float64
hvite_blodlegemer                float64
hjertefrekvens                   float64
respirasjonsfrekvens             float64
kroppstemperatur                 float64
lungefunksjon                    float64
serumalbumin                     float64
bilirubin                        float64
kreatinin                        float64
natrium                          float64
blod_ph                          float64
glukose                          float64
blodurea_nitrogen                float64
urinmengde                       float64
sykdomskategori_id                object
sykdomskategori                   object
dødsfall        

## Kategorisk data til numerisk data

Her setter vi inn dummies for alle object kolonner, ['kjønn', 'inntekt', 'etnisitet', 'sykdomskategori_id', 'sykdomskategori', 'sykdom_underkategori', 'kreft', 'dnr_status']

### Kjønn

In [13]:
df_train['kjønn'].value_counts()

kjønn
male      3074
female    2344
Name: count, dtype: int64

In [14]:
kjønn_encoder = OneHotEncoder(sparse_output=False, drop='first')
kjønn_encoder.fit(df_train[['kjønn']])

def kjønn_dummies(df):
    kjønn_encoded = kjønn_encoder.transform(df[['kjønn']])
    encoded_df = pd.DataFrame(kjønn_encoded, columns=kjønn_encoder.get_feature_names_out(['kjønn']))
    encoded_df.index = df.index
    df = df.drop(columns=['kjønn']).join(encoded_df)
    return df

df_train = kjønn_dummies(df_train)
df_val = kjønn_dummies(df_val)
df_test = kjønn_dummies(df_test)


In [15]:
df_train.head()

,pasient_id,alder,utdanning,inntekt,etnisitet,sykehusdød,oppholdslengde,blodtrykk,hvite_blodlegemer,hjertefrekvens,...,overlevelsesestimat_2mnd,overlevelsesestimat_6mnd,diabetes,demens,kreft,lege_overlevelsesestimat_2mnd,lege_overlevelsesestimat_6mnd,dnr_status,dnr_dag,kjønn_male
2751,3215,86.13599,12.0,NaN,white,0,26,60.0,7.899414,130.0,...,0.282959,0.173981,0,0,no,0.40,0.3,None,NaN,1.0
4704,5522,70.09698,9.0,under $11k,white,0,13,73.0,10.599609,65.0,...,0.858887,0.809937,0,0,no,0.90,0.9,None,NaN,1.0
7059,8304,69.15295,NaN,NaN,white,0,8,102.0,11.000000,66.0,...,0.843994,0.743896,0,0,no,0.80,0.6,None,NaN,1.0
7262,8554,89.88397,8.0,under $11k,white,0,5,61.0,NaN,96.0,...,0.756958,0.614990,0,0,no,0.95,0.6,dnr ved innleggelse,0.0,1.0
6899,8115,76.06598,8.0,NaN,white,0,25,114.0,12.298828,75.0,...,0.312988,0.240997,0,0,no,0.30,0.2,dnr ved innleggelse,0.0,0.0


### Inntekt

In [16]:
df_train['inntekt'].value_counts(dropna=False)

inntekt
NaN           1765
under $11k    1715
$11-$25k       929
$25-$50k       619
>$50k          390
Name: count, dtype: int64

In [17]:
inntekt_mapping = {
            'under $11k': 0, 
           '$11-$25k':1,
           '$25-$50k':2, 
           '>$50k':3
        }

df_train['inntekt'] = df_train['inntekt'].map(inntekt_mapping)
df_val['inntekt'] = df_val['inntekt'].map(inntekt_mapping)
df_test['inntekt'] = df_test['inntekt'].map(inntekt_mapping)

In [18]:
df_train['inntekt'].value_counts(dropna=False)

inntekt
NaN    1765
0.0    1715
1.0     929
2.0     619
3.0     390
Name: count, dtype: int64

## Etnisitet 

In [19]:
 df_train['etnisitet'].value_counts(dropna=False)

etnisitet
white       4274
black        845
hispanic     166
other         60
asian         45
NaN           28
Name: count, dtype: int64

In [20]:
etnisitet_encoder = OneHotEncoder(sparse_output=False)
etnisitet_encoder.fit(df_train[['etnisitet']])

train_etnisitet_dummies = etnisitet_encoder.transform(df_train[['etnisitet']])
val_etnisitet_dummies = etnisitet_encoder.transform(df_val[['etnisitet']])
test_etnisitet_dummies = etnisitet_encoder.transform(df_test[['etnisitet']])

col_names = etnisitet_encoder.get_feature_names_out(['etnisitet'])

train_etnisitet_dummies_df = pd.DataFrame(train_etnisitet_dummies, columns=col_names)
val_etnisitet_dummies_df = pd.DataFrame(val_etnisitet_dummies, columns=col_names)
test_etnisitet_dummies_df = pd.DataFrame(test_etnisitet_dummies, columns=col_names)

df_train = df_train.reset_index(drop=True)
train_etnisitet_dummies_df = train_etnisitet_dummies_df.reset_index(drop=True)

df_val = df_val.reset_index(drop=True)
val_etnisitet_dummies_df = val_etnisitet_dummies_df.reset_index(drop=True)

df_test = df_test.reset_index(drop=True)
test_etnisitet_dummies_df = test_etnisitet_dummies_df.reset_index(drop=True)

df_train = df_train.drop('etnisitet', axis=1).join(train_etnisitet_dummies_df)
df_val = df_val.drop('etnisitet', axis=1).join(val_etnisitet_dummies_df)
df_test = df_test.drop('etnisitet', axis=1).join(test_etnisitet_dummies_df)

In [21]:
df_train.head()

,pasient_id,alder,utdanning,inntekt,sykehusdød,oppholdslengde,blodtrykk,hvite_blodlegemer,hjertefrekvens,respirasjonsfrekvens,...,lege_overlevelsesestimat_6mnd,dnr_status,dnr_dag,kjønn_male,etnisitet_asian,etnisitet_black,etnisitet_hispanic,etnisitet_other,etnisitet_white,etnisitet_nan
0,3215,86.13599,12.0,NaN,0,26,60.0,7.899414,130.0,26.0,...,0.3,None,NaN,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,5522,70.09698,9.0,0.0,0,13,73.0,10.599609,65.0,7.0,...,0.9,None,NaN,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,8304,69.15295,NaN,NaN,0,8,102.0,11.000000,66.0,24.0,...,0.6,None,NaN,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,8554,89.88397,8.0,0.0,0,5,61.0,NaN,96.0,20.0,...,0.6,dnr ved innleggelse,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,8115,76.06598,8.0,NaN,0,25,114.0,12.298828,75.0,32.0,...,0.2,dnr ved innleggelse,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## sykdomskategori_id, sykdomskategori and sykdom_underkategori

In [22]:
df_train['sykdomskategori_id'].value_counts(dropna=False)

sykdomskategori_id
A1s    2518
BrY    1701
ChE     841
DWw     358
Name: count, dtype: int64

In [23]:
df_train['sykdomskategori'].value_counts(dropna=False)

sykdomskategori
ARF/MOSF              2518
COPD/CHF/Cirrhosis    1701
Cancer                 841
Coma                   358
Name: count, dtype: int64

In [24]:
df_train['sykdom_underkategori'].value_counts(dropna=False)

sykdom_underkategori
ARF/MOSF w/Sepsis    2087
CHF                   855
COPD                  584
Lung Cancer           542
MOSF w/Malig          431
Coma                  358
Colon Cancer          299
Cirrhosis             262
Name: count, dtype: int64

Vi ser over at sykdomskategori_id og sykdomskategori har likt antall verdier. Vi kan derfor prøve å utforske om disse kolonnene er like, og om vi eventuelt kan fjerne en. Vi starter denne hypotesen med å sjekke om alle verider er likke for begge kolonnene: 

In [25]:
sykdom_map = {
    'A1s':'ARF/MOSF', 
    'BrY':'COPD/CHF/Cirrhosis',
    'ChE':'Cancer', 
    'DWw':'Coma'
    }

print(all(df_train['sykdomskategori_id'].map(sykdom_map) == df_train['sykdomskategori']))

True


Vi vet nå at disse to kolonnene er like. Dette betyr at vi fint kan fjerne en av de, men vi har en kolonne til, sykdom_underkategori. Hvis hver underkategori er bestemt av dens overkategori, vil sykdom_underkategori gi mer informasjon, og vi kan droppe sykdomskategori og sykdomskategori_id. 

In [26]:
#sjekker verdi overlapp mellom id og underkategori
sammenheng = df_train.groupby('sykdomskategori_id')['sykdom_underkategori'].unique()
print(sammenheng)

sykdomskategori_id
A1s    [ARF/MOSF w/Sepsis, MOSF w/Malig]
BrY               [COPD, CHF, Cirrhosis]
ChE          [Lung Cancer, Colon Cancer]
DWw                               [Coma]
Name: sykdom_underkategori, dtype: object


Som vi kan se er det ingen overlapping i verdiene. Hvis vi for eksempel har en av underkategoriene i kreft (lung eller colon), står det alltid ChE i kolonnen sykdomskategori_id. Dette forteller oss at vi trygt kan slette sykdomskategori_id og sykdomskategori da de gir smindre informasjon enn sykdom_underkategori. 

Fortsetter med å sette inn dummies for hver kolonne i sykdom_underkategori og fjerner sykdomskategori_id og sykdomskategori.

In [27]:
sykdom_encoder = OneHotEncoder(sparse_output=False)
sykdom_encoder.fit(df_train[['sykdom_underkategori']])

train_dummies = sykdom_encoder.transform(df_train[['sykdom_underkategori']])
val_dummies = sykdom_encoder.transform(df_val[['sykdom_underkategori']])
test_dummies = sykdom_encoder.transform(df_test[['sykdom_underkategori']])

underkategori = sykdom_encoder.get_feature_names_out(['sykdom_underkategori'])

train_dummies_df = pd.DataFrame(train_dummies, columns=underkategori, index=df_train.index)
val_dummies_df = pd.DataFrame(val_dummies, columns=underkategori, index=df_val.index)
test_dummies_df = pd.DataFrame(test_dummies, columns=underkategori, index=df_test.index)

df_train = df_train.drop(['sykdomskategori', 'sykdomskategori_id', 'sykdom_underkategori'], axis=1)
df_val = df_val.drop(['sykdomskategori', 'sykdomskategori_id', 'sykdom_underkategori'], axis=1)
df_test = df_test.drop(['sykdomskategori', 'sykdomskategori_id', 'sykdom_underkategori'], axis=1)

df_train = df_train.join(train_dummies_df)
df_val = df_val.join(val_dummies_df)
df_test = df_test.join(test_dummies_df)

## Kreft

In [28]:
 df_train['kreft'].value_counts(dropna=False)

kreft
no            3563
metastatic    1102
yes            753
Name: count, dtype: int64

In [29]:
kreft_mapping = {
    'no': 0, 
    'yes': 1,
    'metastatic': 2, 
    }

df_train['kreft'] = df_train['kreft'].map(kreft_mapping)
df_val['kreft'] = df_val['kreft'].map(kreft_mapping)
df_test['kreft'] = df_test['kreft'].map(kreft_mapping)

In [30]:
 df_train['kreft'].value_counts(dropna=False)

kreft
0    3563
2    1102
1     753
Name: count, dtype: int64

## dnr_status

In [31]:
df_train['dnr_status'].value_counts(dropna=False)

dnr_status
None                   4669
dnr ved innleggelse     594
dnr før innleggelse     155
Name: count, dtype: int64

In [32]:
dnr_mapping = {
    'none': 0,
    'dnr ved innleggelse': 1, 
    'dnr før innleggelse': 2}

df_train['dnr_status'] = df_train['dnr_status'].map(dnr_mapping).fillna(0)
df_val['dnr_status'] = df_val['dnr_status'].map(dnr_mapping).fillna(0)
df_test['dnr_status'] = df_test['dnr_status'].map(dnr_mapping).fillna(0)

In [33]:
df_train['dnr_status'].value_counts()

dnr_status
0.0    4669
1.0     594
2.0     155
Name: count, dtype: int64

In [34]:
df_train.dtypes

pasient_id                                  int64
alder                                     float64
utdanning                                 float64
inntekt                                   float64
sykehusdød                                  int64
oppholdslengde                              int64
blodtrykk                                 float64
hvite_blodlegemer                         float64
hjertefrekvens                            float64
respirasjonsfrekvens                      float64
kroppstemperatur                          float64
lungefunksjon                             float64
serumalbumin                              float64
bilirubin                                 float64
kreatinin                                 float64
natrium                                   float64
blod_ph                                   float64
glukose                                   float64
blodurea_nitrogen                         float64
urinmengde                                float64


Vi ser over at vi har fått omgjort alle kolonner med typen object. 

## Verdier målt senere enn dag 1

Siden vi skal beregne oppholdslengde fra dag 1 fjerner jeg bilirubin og adl_pasient, ettersom vi får oppgitt at denne dataen beregnes fra dag 7

In [35]:
fjern_kolonner = ['bilirubin', 'adl_pasient']

df_train = df_train.drop(columns=fjern_kolonner, errors='ignore')
df_val = df_val.drop(columns=fjern_kolonner, errors='ignore')
df_test = df_test.drop(columns=fjern_kolonner, errors='ignore')

Sykehusdød og dødsfall gir oss informasjon om en pasient dør på sykehuset eller etter. Ettersom vi skal finne oppholdslengde fra dag 1 er dette to kolonner vi kan fjerne

In [36]:
fjern_kolonner = ['sykehusdød', 'dødsfall']

df_train = df_train.drop(columns=fjern_kolonner, errors='ignore')
df_val = df_val.drop(columns=fjern_kolonner, errors='ignore')
df_test = df_test.drop(columns=fjern_kolonner, errors='ignore')

In [37]:
df_train

,pasient_id,alder,utdanning,inntekt,oppholdslengde,blodtrykk,hvite_blodlegemer,hjertefrekvens,respirasjonsfrekvens,kroppstemperatur,...,etnisitet_white,etnisitet_nan,sykdom_underkategori_ARF/MOSF w/Sepsis,sykdom_underkategori_CHF,sykdom_underkategori_COPD,sykdom_underkategori_Cirrhosis,sykdom_underkategori_Colon Cancer,sykdom_underkategori_Coma,sykdom_underkategori_Lung Cancer,sykdom_underkategori_MOSF w/Malig
0,3215,86.13599,12.0,NaN,26,60.0,7.899414,130.0,26.0,38.00000,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5522,70.09698,9.0,0.0,13,73.0,10.599609,65.0,7.0,36.50000,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,8304,69.15295,NaN,NaN,8,102.0,11.000000,66.0,24.0,35.69531,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,8554,89.88397,8.0,0.0,5,61.0,NaN,96.0,20.0,36.39844,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,8115,76.06598,8.0,NaN,25,114.0,12.298828,75.0,32.0,39.09375,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5413,6141,38.51297,12.0,1.0,5,83.0,11.199219,78.0,28.0,35.39844,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5414,6336,73.81000,6.0,0.0,5,103.0,7.599609,189.0,30.0,36.19531,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5415,994,38.94299,14.0,3.0,12,108.0,17.699219,115.0,28.0,36.00000,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5416,8953,58.44800,12.0,NaN,16,71.0,24.296875,119.0,32.0,36.79688,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## Lagrer data i filer

In [38]:
X_train = df_train.drop(['pasient_id', 'oppholdslengde'], axis=1)
y_train = df_train.loc[:,'oppholdslengde']

X_val = df_val.drop(['pasient_id', 'oppholdslengde'], axis=1)
y_val = df_val.loc[:,'oppholdslengde']

X_test = df_test.drop(['pasient_id', 'oppholdslengde'], axis=1)
y_test = df_test.loc[:,'oppholdslengde']

data = [(X_train, 'X_train'), (y_train, 'y_train'), (X_val, 'X_val'), 
            (y_val, 'y_val'), (X_test, 'X_test'), (y_test, 'y_test')]

for data, name in data:
    data.to_csv(f'cleaned_data/{name}.csv', index=False)